## Testing file for imaging and dataset generation pipeline

1. **X-ray Simulation** (gvxr) - Generate synthetic X-ray projections from bone + flesh STL meshes at multiple gantry angles
2. **Keypoint Detection** (MMPose) - Detect hand bone keypoints in each X-ray image
3. **Segmentation** (SAM) - Use keypoint prompts to segment bone regions from the X-ray images
4. **Per-View 3D Mesh** (TripoSR) - Reconstruct a 3D mesh from each segmented single-view X-ray

## Move to original source folder before the execution!

In [ ]:
!pip install torch==2.1.2+cu118 torchvision==0.16.2+cu118 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu118

In [ ]:
!pip install -r ./mmpose/requirements.txt

In [ ]:
!pip install segment_anything

In [ ]:
!pip install -U openmim
!pip install numpy==1.26.4
!pip install mmengine==0.10.3
!mim install mmcv-2.1.0-cp39-cp39-win_amd64.whl
!mim install "mmdet>=3.1.0"
!mim install mmpose

In [ ]:
!pip install gvxr 

In [ ]:
!pip install numpy==1.26.4

In [ ]:
!pip install pybind11
!pip install scikit_build_core
!pip install Cargo

In [ ]:
!pip install "cmake>=3.15"
!pip install -r requirements.txt --y
!pip install torchmcubes --no-build-isolation

In [1]:
# Basic Setup

# Import packages
import os
import numpy as np
import torch, mmcv
print(torch.__version__)        # should be 2.1.0+cu118
print(mmcv.__version__)         # should be 2.1.0
!python -c "import numpy, xtcocotools, torch, mmcv; print(numpy.__version__, torch.__version__, mmcv.__version__)"

#should be
#2.1.2+cu118
#2.1.0
#1.26.4 2.1.2+cu118 2.1.0

has_mpl = True
try:
    import matplotlib # To plot images
    import matplotlib.pyplot as plt # Plotting
    from matplotlib.colors import LogNorm # Look up table
    from matplotlib.colors import PowerNorm # Look up table

    font = {'family' : 'serif',
             'size'   : 15
           }
    matplotlib.rc('font', **font)

    # Uncomment the line below to use LaTeX fonts
    # matplotlib.rc('text', usetex=True)
except:
    has_mpl = False
#print(dir(gvxr))
# from tifffile import imwrite # Write TIFF files

import torch
import cv2
from segment_anything import sam_model_registry, SamPredictor
from glob import glob
from mmpose.apis import init_model, inference_topdown
from mmpose.structures import merge_data_samples
from mmpose.registry import VISUALIZERS
import mmcv
from tifffile import imwrite
from PIL import Image

# Load config and checkpoint
# === CONFIGURATION ===
DATASET_DIR = "./Mura/"
RESULT_DIR = "./New_Result/"
MASK_DIR = os.path.join(RESULT_DIR, "Mask")

ORIGINAL_DIR = os.path.join(RESULT_DIR, "Original")
PROMPT_DIR = os.path.join(RESULT_DIR, "Prompt")
PREPROCESS_DIR = os.path.join(RESULT_DIR, "Preprocessed")
os.makedirs(MASK_DIR, exist_ok=True)
os.makedirs(ORIGINAL_DIR, exist_ok=True)
os.makedirs(PROMPT_DIR, exist_ok=True)
os.makedirs(PREPROCESS_DIR, exist_ok=True)

# === Load SAM2 Model ===
MODEL_TYPE = "vit_h"  
CHECKPOINT_PATH = "d:/Jupyter/3D_Image_segmentation/sam_vit_h.pth"  
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

sam = sam_model_registry[MODEL_TYPE](checkpoint=CHECKPOINT_PATH).to(DEVICE)
predictor = SamPredictor(sam)

# === Load mmpose Model ===
config_file = 'mmpose/configs/hand_2d_keypoint/rtmpose/hand5/rtmpose-m_8xb256-210e_hand5-256x256.py'
checkpoint_file = 'rtmpose-m_simcc-hand5_pt-aic-coco_210e-256x256-74fb594_20230320.pth'  # download from mmpose model zoo

pose_model = init_model(config_file, checkpoint_file, device=DEVICE)

HAND5_NAMES = [
    "wrist",
    "thumb1","thumb2","thumb3","thumb4",
    "index1","index2","index3","index4",
    "middle1","middle2","middle3","middle4",
    "ring1","ring2","ring3","ring4",
    "pinky1","pinky2","pinky3","pinky4"
]

# Set Environment Variables for cuDNN
os.environ['CUDNN_INCLUDE_DIR'] = r'C:\Program Files\NVIDIA\CUDNN\v9.2\include'
os.environ['CUDNN_LIB_DIR'] = r'C:\Program Files\NVIDIA\CUDNN\v9.2\lib\12.5\x64'

# Verify the Environment Variables
print("CUDNN_INCLUDE_DIR:", os.environ.get('CUDNN_INCLUDE_DIR'))
print("CUDNN_LIB_DIR:", os.environ.get('CUDNN_LIB_DIR'))

2.1.2+cu118
2.1.0
1.26.4 2.1.2+cu118 2.1.0
Loads checkpoint by local backend from path: rtmpose-m_simcc-hand5_pt-aic-coco_210e-256x256-74fb594_20230320.pth
CUDNN_INCLUDE_DIR: C:\Program Files\NVIDIA\CUDNN\v9.2\include
CUDNN_LIB_DIR: C:\Program Files\NVIDIA\CUDNN\v9.2\lib\12.5\x64


In [2]:
#mmpose setup

def hand_prompts(
    keypoints, scores, image_shape=None,
    drop_label4=True,
    alpha_thumb=0.50,           # wrist <-> thumb1
    alpha_index_from_thumb=0.55,# thumb_interp <-> index1   (special case)
    alpha_middle=0.45,          # wrist <-> middle1
    alpha_ring=0.45,            # wrist <-> ring1
    alpha_pinky=0.45,            # wrist <-> pinky1
    add_external_negatives=True,  # NEW PARAMETER
    negative_margin=50,  # NEW: pixels outside hand bbox
    extrapolate_tips = 0.1
):
    """
    keypoints: (21,2) or (1,21,2), (x,y)
    scores:    (21,)  or (1,21)
    Returns:
      pos_coords (M,2), pos_names (list[str]), pos_scores (M,)
      neg_coords (K,2), neg_names (list[str])   # the dropped label-4 tips
    """
    # Accept both shapes from MMPose
    kps = keypoints[0] if keypoints.ndim == 3 else keypoints
    scs = scores[0]    if scores.ndim == 2 else scores

    H, W = (image_shape if image_shape is not None else (None, None))
    def _clip(pt):
        if H is None or W is None: return pt.astype(np.float32)
        return np.array([np.clip(pt[0], 0, W-1), np.clip(pt[1], 0, H-1)], dtype=np.float32)

    # indices
    WRIST = 0
    TH1, IN1, MI1, RI1, PI1 = 1, 5, 9, 13, 17
    tips = [4, 8, 12, 16, 20]  # label-4 to remove
    TH3, IN3, MI3, RI3, PI3 = 3, 7, 11, 15, 19

    # 1) keep all originals except the label-4 tips (wrist is always kept)
    keep_mask = np.ones(21, dtype=bool)
    if drop_label4:
        keep_mask[tips] = False
    keep_mask[WRIST] = True

    if extrapolate_tips > 0 and drop_label4:
        # Extrapolate each fingertip beyond its actual position
        # Using the direction from the 3rd joint (label-3) to the tip (label-4)
        tip_base_pairs = [
            (TH3, 4, "thumb"),   # thumb: joint3 -> tip
            (IN3, 8, "index"),   # index: joint3 -> tip
            (MI3, 12, "middle"), # middle: joint3 -> tip
            (RI3, 16, "ring"),   # ring: joint3 -> tip
            (PI3, 20, "pinky")   # pinky: joint3 -> tip
        ]
        
        neg_coords_list = []
        neg_names = []
        
        for base_idx, tip_idx, finger_name in tip_base_pairs:
            base_pt = kps[base_idx].astype(np.float32)
            tip_pt = kps[tip_idx].astype(np.float32)
            
            # Calculate direction vector from base to tip
            direction = tip_pt - base_pt
            
            # Extrapolate beyond tip
            extrapolated_tip = tip_pt + direction * extrapolate_tips
            
            neg_coords_list.append(_clip(extrapolated_tip))
            neg_names.append(f"{finger_name}_tip_extrapolated")
    else:
        # Original behavior: use actual tips as negatives
        neg_coords_list = [kps[i] for i in tips] if drop_label4 else []
        neg_names = [HAND5_NAMES[i] for i in tips] if drop_label4 else []


    pos_coords = [kps[i] for i in range(21) if keep_mask[i]]
    pos_scores = [scs[i] for i in range(21) if keep_mask[i]]
    pos_names  = [HAND5_NAMES[i] for i in range(21) if keep_mask[i]]

    wrist_pt, wrist_sc = kps[WRIST].astype(np.float32), float(scs[WRIST])

    # 2) new interpolated points
    def _interp(p, q, a): return _clip((1.0 - a) * p + a * q)

    thumb_pt = kps[TH1].astype(np.float32)
    thumb_sc = float(scs[TH1])
    thumb_interp = _interp(wrist_pt, thumb_pt, alpha_thumb)
    thumb_interp_sc = 0.5 * (wrist_sc + thumb_sc)

    index_pt = kps[IN1].astype(np.float32)
    index_sc = float(scs[IN1])
    index_interp = _interp(thumb_interp, index_pt, alpha_index_from_thumb)  # special chain
    index_interp_sc = 0.5 * (thumb_interp_sc + index_sc)

    middle_pt = kps[MI1].astype(np.float32)
    ring_pt   = kps[RI1].astype(np.float32)
    pinky_pt  = kps[PI1].astype(np.float32)
    middle_interp = _interp(wrist_pt, middle_pt, alpha_middle)
    ring_interp   = _interp(wrist_pt, ring_pt,   alpha_ring)
    pinky_interp  = _interp(wrist_pt, pinky_pt,  alpha_pinky)


    thumb_tip_sw = _interp(kps[TH3].astype(np.float32), kps[4].astype(np.float32), 0.5)
    index_tip_sw = _interp(kps[IN3].astype(np.float32), kps[8].astype(np.float32), 0.5)
    middle_tip_sw = _interp(kps[MI3].astype(np.float32), kps[12].astype(np.float32), 0.5)
    ring_tip_sw = _interp(kps[RI3].astype(np.float32), kps[16].astype(np.float32), 0.5)
    pinky_tip_sw = _interp(kps[PI3].astype(np.float32), kps[20].astype(np.float32), 0.5)

    middle_interp_sc = 0.5 * (wrist_sc + float(scs[MI1]))
    ring_interp_sc   = 0.5 * (wrist_sc + float(scs[RI1]))
    pinky_interp_sc  = 0.5 * (wrist_sc + float(scs[PI1]))

    # append 5 new points
    new_points = [
        (thumb_interp,  "thumb_interp",  thumb_interp_sc),
        (index_interp,  "index_interp",  index_interp_sc),
        (middle_interp, "middle_interp", middle_interp_sc),
        (ring_interp,   "ring_interp",   ring_interp_sc),
        (pinky_interp,  "pinky_interp",  pinky_interp_sc),
        (thumb_tip_sw,  "thumb_interp2",  0.5),
        (index_tip_sw,  "index_interp2",  0.5),
        (middle_tip_sw, "middle_interp2", 0.5),
        (ring_tip_sw,   "ring_interp2",   0.5),
        (pinky_tip_sw,  "pinky_interp2",  0.5)
    ]
    for pt, name, sc in new_points:
        pos_coords.append(pt)
        pos_names.append(name)
        pos_scores.append(sc)

    if add_external_negatives and H is not None and W is not None:
        # Calculate hand bounding box
        all_x = kps[:, 0]
        all_y = kps[:, 1]
        x_min, x_max = np.min(all_x), np.max(all_x)
        y_min, y_max = np.min(all_y), np.max(all_y)
        
        # Add negative points outside the hand bbox
        external_negatives = []
        
        # Top side (above hand)
        if y_min - negative_margin > 0:
            external_negatives.append(
                (np.array([(x_min + x_max) / 2, y_min - negative_margin]), "neg_top")
            )
        
        # Bottom side (below wrist)
        '''if y_max + negative_margin < H:
            external_negatives.append(
                (np.array([(x_min + x_max) / 2, y_max + negative_margin]), "neg_bottom")
            )'''
        
        # Left side
        if x_min - negative_margin > 0:
            external_negatives.append(
                (np.array([x_min - negative_margin, (y_min + y_max) / 2]), "neg_left")
            )
        
        # Right side
        if x_max + negative_margin < W:
            external_negatives.append(
                (np.array([x_max + negative_margin, (y_min + y_max) / 2]), "neg_right")
            )
        
        # Corners (optional, more aggressive)
        if y_min - negative_margin > 0 and x_min - negative_margin > 0:
            external_negatives.append(
                (np.array([x_min - negative_margin, y_min - negative_margin]), "neg_topleft")
            )
        if y_min - negative_margin > 0 and x_max + negative_margin < W:
            external_negatives.append(
                (np.array([x_max + negative_margin, y_min - negative_margin]), "neg_topright")
            )
        
        # Add to negative lists
        for pt, name in external_negatives:
            neg_coords_list.append(_clip(pt))
            neg_names.append(name)

    pos_coords = np.array(pos_coords, dtype=np.float32)
    pos_scores = np.array(pos_scores, dtype=np.float32)
    neg_coords = np.array(neg_coords_list, dtype=np.float32) if neg_coords_list else np.empty((0,2), np.float32)

    return pos_coords, pos_names, pos_scores, neg_coords, neg_names


In [3]:
# file name setup

def file_name(base_name, directory=".", extension=".png"):
    # Ensure directory exists
    if not os.path.exists(directory):
        os.makedirs(directory)

    # Start the counter at 1 (or 0 if you prefer)
    counter = 1
    
    while True:
        # :06d formats the integer to 6 digits with leading zeros
        suffix = f"{counter:06d}"
        
        filename = f"{suffix}_{base_name}_{extension}"
        file_path = os.path.join(directory, filename)
        
        if os.path.exists(file_path):
            # If file exists, try the next number
            counter += 1
        else:
            # If file is free, return this path
            return file_path, filename

In [4]:
import math
from gvxrPython3 import gvxr # Simulate X-ray images

def diagnose_setup():
    """Check the current gVXR setup and object positions"""
    print("\n=== DIAGNOSTIC INFO ===")
    
    # Get scene bounding box
    try:
        # Check if objects are loaded
        node_labels = gvxr.getChildrenLabels("root")
        print(f"Loaded objects: {node_labels}")
        
        for label in node_labels:
            if label and label != "root":
                bbox = gvxr.getNodeBoundingBox(label, "cm")
                print(f"  {label} bounding box: {bbox}")
    except Exception as e:
        print(f"Could not get node info: {e}")
    
    # Try to get scene info
    try:
        scene_bbox = gvxr.getSceneBoundingBox("cm")
        print(f"Scene bounding box: {scene_bbox}")
    except Exception as e:
        print(f"Could not get scene bounding box: {e}")


# ============================================================
# FIXED GANTRY POSITIONING
# ============================================================
def rotate_point_around_y_axis(x, y, z, angle_degrees):
    """
    Rotate a point (x, y, z) around the Y-axis by the given angle.
    
    Args:
        x, y, z: Original coordinates
        angle_degrees: Rotation angle in degrees (positive = counterclockwise when viewed from above)
    
    Returns:
        (new_x, new_y, new_z): Rotated coordinates
    """
    angle_rad = math.radians(angle_degrees)
    
    # Rotation matrix around Y-axis:
    # | cos(theta)   0   sin(theta) |   | x |
    # |   0      1     0    | x | y |
    # | -sin(theta)  0   cos(theta) |   | z |
    
    new_x = x * math.cos(angle_rad) + y * math.sin(angle_rad)
    new_y = -x * math.sin(angle_rad) + y * math.cos(angle_rad)
    new_z = z  # Y doesn't change
    
    return new_x, new_y, new_z

def setup_gantry_at_angle(angle_degrees, source_distance=-50.0, detector_distance=50.0, verbose = False, scene_rotate = [0, 0, 0]):
    """
    Position the X-ray source and detector at a specific angle.
    
    This version ensures proper detector orientation.
    
    Coordinate system:
    - X-axis: Left-Right
    - Y-axis: Up-Down (vertical)
    - Z-axis: Front-Back
    
    At 0 deg: Source behind object (-Z), Detector in front (+Z) - AP view
    """
    
    
    # For a standard AP (anterior-posterior) setup rotating around Y-axis:
    # At 0 deg: Source at (0, 0, -distance), Detector at (0, 0, +distance)
    
    base_source = (source_distance, 0.0, 0.0)
    base_detector = (detector_distance, 0.0, 0.0)

    source_x, source_y, source_z = rotate_point_around_y_axis(
        base_source[0], base_source[1], base_source[2], angle_degrees
    )
    detector_x, detector_y, detector_z = rotate_point_around_y_axis(
        base_detector[0], base_detector[1], base_detector[2], angle_degrees
    )
    
    # Set positions
    gvxr.setSourcePosition(source_x, source_y, source_z, "cm")
    gvxr.setDetectorPosition(detector_x, detector_y, detector_z, "cm")
    
    # Detector up vector should always point in Y direction
    gvxr.setDetectorUpVector(0, 0, -1)
    if len(scene_rotate) !=3:
        raise TypeError("Wrong input for rotation values")
    gvxr.rotateScene(scene_rotate[0], scene_rotate[1], scene_rotate[2])
    
    if verbose:
        print(f"Gantry angle: {angle_degrees} deg")
        print(f"  Source: ({source_x:.2f}, {source_y:.2f}, {source_z:.2f}) cm")
        print(f"  Detector: ({detector_x:.2f}, {detector_y:.2f}, {detector_z:.2f}) cm")


# ============================================================
# ALTERNATIVE: ROTATE SCENE INSTEAD OF GANTRY
# ============================================================

def setup_fixed_gantry(SourcePosition = -50, DetectorPosition = 50):
    """Set up a fixed source-detector pair along X-axis"""
    gvxr.setSourcePosition(SourcePosition, 0.0, 0.0, "cm")
    gvxr.setDetectorPosition(DetectorPosition, 0.0, 0.0, "cm")
    gvxr.setDetectorUpVector(0, 0, -1)
    print("Fixed gantry: Source at (-50, 0, 0), Detector at (50, 0, 0)")


def capture_by_rotating_scene(angle_degrees):
    """
    Capture projection by rotating the scene instead of moving the gantry.
    This is often more reliable in gVXR.
    """
    # Reset scene transformation
    gvxr.resetSceneTransformation()
    
    # Center the scene
    gvxr.moveToCentre()
    
    # Rotate scene around Y-axis (vertical)
    # Negative angle because we want the equivalent of rotating the gantry
    gvxr.rotateScene(0, -angle_degrees, 0)
    
    print(f"Scene rotated: {-angle_degrees} deg around Y-axis")
    
    # Compute image
    projections = np.array(gvxr.computeXRayImage()).astype(np.float32)
    
    return projections

In [5]:
#SAM2 Setup

def process_single_image(image_path, image_name, pose_model, predictor, 
                         MASK_DIR, ORIGINAL_DIR, PREPROCESS_DIR, PROMPT_DIR, MASKED_DIR, mask_num = 1,
                         show_plots=False):
    """
    Process a single X-ray image through pose estimation and SAM segmentation.
    
    Args:
        image_path: Path to the input image
        image_name: Name of the image file
        pose_model: MMPose model for hand keypoint detection
        predictor: SAM predictor
        MASK_DIR, ORIGINAL_DIR, PREPROCESS_DIR, PROMPT_DIR, MASKED_DIR: Output directories
        show_plots: Whether to display plots (set False for batch processing)
    
    Returns:
        dict: Paths to saved files and results
    """
    
    # Load image
    image = cv2.imread(image_path)
    if image is None:
        print(f"  ERROR: Could not load {image_path}")
        return None
    
    height, width = image.shape[:2]
    
    # Create bounding box for full image
    hand_bboxes = [{
        'bbox': np.array([0, 0, width, height], dtype=np.float32),
        'bbox_score': 1.0
    }]
    
    # Run pose estimation
    hand_bboxes_xywh = [hand['bbox'] for hand in hand_bboxes]
    pose_results = inference_topdown(pose_model, image, hand_bboxes_xywh, bbox_format='xywh')
    
    keypoints = pose_results[0].pred_instances.keypoints
    scores = pose_results[0].pred_instances.keypoint_scores
    
    # Generate hand prompts
    kps = keypoints
    scs = scores
    
    pos_coords, pos_names, pos_scores, neg_coords, neg_names = hand_prompts(
        kps, scs, image_shape=image.shape[:2],
        drop_label4=True,
        alpha_thumb=0.50,
        alpha_index_from_thumb=0.55,
        alpha_middle=0.45,
        alpha_ring=0.45,
        alpha_pinky=0.45
    )
    
    # Build SAM prompts
    point_coords = np.vstack([pos_coords, neg_coords])
    point_labels = np.hstack([
        np.ones(len(pos_coords), dtype=np.int32),
        np.zeros(len(neg_coords), dtype=np.int32)
    ])
    
    # Optional: Plot keypoints
    if show_plots:
        plt.figure(figsize=(8, 8))
        plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        for i in range(point_coords.shape[0]):
            x, y = point_coords[i, :]
            if point_labels[i] == 1:
                plt.scatter(x, y, c='r', s=20)
        plt.axis('off')
        #plt.title(f'Hand Keypoints - {image_name}')
        plt.show()
    
    # Run SAM segmentation
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    predictor.set_image(image_rgb)
    
    input_points_np = np.array(point_coords)
    input_labels_np = np.array(point_labels)
    
    masks, scores_sam, _ = predictor.predict(
        point_coords=input_points_np,
        point_labels=input_labels_np,
        multimask_output=True
    )
    
    # Select mask
    final_mask = masks[mask_num]
    
    # Optional: Plot mask overlay
    if show_plots:
        plt.figure(figsize=(8, 8))
        plt.imshow(image_rgb)
        masked_overlay = np.zeros_like(image_rgb)
        masked_overlay[final_mask] = [0, 255, 0]
        plt.imshow(masked_overlay, alpha=0.4)
        
        added, subtracted = 0, 0
        for i in range(len(input_points_np)):
            if input_labels_np[i] == 0:
                label = "Prompts subtracted" if subtracted == 0 else None
                plt.scatter(input_points_np[i, 0], input_points_np[i, 1], c='red', s=30, label=label)
                subtracted += 1
            else:
                label = "Prompts added" if added == 0 else None
                plt.scatter(input_points_np[i, 0], input_points_np[i, 1], c='blue', s=30, label=label)
                added += 1
        plt.legend()
        plt.axis('off')
        #plt.title(f"SAM Mask Overlay - {image_name}")
        plt.show()
    
    # Define save paths
    base_name = os.path.splitext(image_name)[0]
    mask_save_path = os.path.join(MASK_DIR, f"{base_name}_mask.png")
    orig_save_path = os.path.join(ORIGINAL_DIR, f"{base_name}_orig.png")
    pre_save_path = os.path.join(PREPROCESS_DIR, f"{base_name}_preprocessed.png")
    prompt_save_path = os.path.join(PROMPT_DIR, f"{base_name}.txt")
    masked_save_path = os.path.join(MASKED_DIR, f"{base_name}_masked.png")
    
    # Save outputs
    final_mask_uint8 = final_mask.astype(np.uint8) * 255
    cv2.imwrite(mask_save_path, final_mask_uint8)
    cv2.imwrite(pre_save_path, cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR))
    cv2.imwrite(orig_save_path, image)
    np.savetxt(prompt_save_path, np.column_stack((input_points_np, input_labels_np)), 
               fmt='%d', delimiter=',', header="x,y,label")
    
    # Create masked RGBA image
    image_pil = Image.open(orig_save_path)
    mask_pil = Image.open(mask_save_path).convert("L")
    
    image_np = np.array(image_pil)
    mask_np = np.array(mask_pil)
    binary_mask = (mask_np > 0).astype(np.uint8) * 255
    
    if image_np.ndim == 2:
        rgb_image = np.stack([image_np] * 3, axis=-1)
    else:
        rgb_image = image_np[:, :, :3]
    
    rgba_image = np.zeros((rgb_image.shape[0], rgb_image.shape[1], 4), dtype=np.uint8)
    rgba_image[:, :, :3] = rgb_image
    rgba_image[:, :, 3] = binary_mask
    
    masked_image_pil = Image.fromarray(rgba_image, mode='RGBA')
    masked_image_pil.save(masked_save_path)
    
    # Optional: Display masked result
    if show_plots:
        plt.figure(figsize=(8, 8))
        plt.imshow(masked_image_pil)
        plt.axis('off')
        #plt.title(f"Masked Result - {image_name}")
        plt.show()
    
    return {
        'mask_path': mask_save_path,
        'orig_path': orig_save_path,
        'preprocessed_path': pre_save_path,
        'prompt_path': prompt_save_path,
        'masked_path': masked_save_path,
        'keypoints': keypoints,
        'mask': final_mask
    }


# ============================================================
# BATCH PROCESSING FUNCTION
# ============================================================

def process_all_images(original_image_paths, pose_model, predictor,
                       MASK_DIR, ORIGINAL_DIR, PREPROCESS_DIR, PROMPT_DIR, MASKED_DIR, mask_num = 1,
                       show_plots=False, verbose = False):
    """
    Process all images in the list through pose estimation and SAM segmentation.
    
    Args:
        original_image_paths: List of paths to input images
        pose_model: MMPose model
        predictor: SAM predictor
        MASK_DIR, ORIGINAL_DIR, PREPROCESS_DIR, PROMPT_DIR, MASKED_DIR: Output directories
        show_plots: Whether to display plots
    
    Returns:
        dict: {image_path: results} for all processed images
    """
    
    # Create output directories
    for dir_path in [MASK_DIR, ORIGINAL_DIR, PREPROCESS_DIR, PROMPT_DIR, MASKED_DIR]:
        os.makedirs(dir_path, exist_ok=True)
    
    all_results = {}
    total = len(original_image_paths)
    
    if verbose:
        print(f"Processing {total} images")
    
    for idx, image_path in enumerate(original_image_paths):
        image_name = os.path.basename(image_path)
        
        if verbose:
            print(f"\n[{idx+1}/{total}] Processing: {image_name}")
        
        try:
            result = process_single_image(
                image_path=image_path,
                image_name=image_name,
                pose_model=pose_model,
                predictor=predictor,
                MASK_DIR=MASK_DIR,
                ORIGINAL_DIR=ORIGINAL_DIR,
                PREPROCESS_DIR=PREPROCESS_DIR,
                PROMPT_DIR=PROMPT_DIR,
                MASKED_DIR=MASKED_DIR,
                mask_num = mask_num,
                show_plots=show_plots
            )
            
            if result:
                all_results[image_path] = result
                if verbose:
                    print(f"  [OK] Saved: {result['masked_path']}")
            else:
                print(f"  [FAIL] Failed to process")
                
        except Exception as e:
            print(f"  [FAIL] Error: {str(e)}")
            continue
    
    if verbose:
        print(f"COMPLETE: {len(all_results)}/{total} images processed successfully")
    
    return all_results

In [6]:
import subprocess
import shutil
import re
from collections import defaultdict


# TripoSR Setup
    
def group_results_by_prefix(results):
    """
    Group results from process_all_images() by common prefix.
    
    Args:
        results: Dict from process_all_images()
                 {original_path: {'masked_path': ..., 'mask_path': ..., ...}}
    
    Returns:
        dict: {prefix: {angle: {'original_path': ..., 'masked_path': ..., ...}}}
    """
    groups = defaultdict(dict)
    
    for original_path, result in results.items():
        if result is None:
            continue
            
        filename = os.path.basename(original_path)
        
        # Pattern: {prefix}_{angle}__ori.png
        # Example: 000003_gvxr_processed_0__ori.png
        match = re.match(r'(.+)_(-?\d+)_(ori|norm|Log|Power|gamma)\.png', filename)
        
        if match:
            prefix = match.group(1)  # '000003_gvxr_processed'
            angle = int(match.group(2))  # 0, 30, -30
            img_type = match.group(3)

            groups[prefix][angle] = {
                'original_path': original_path,
                'masked_path': result['masked_path'],
                'mask_path': result.get('mask_path'),
                'prompt_path': result.get('prompt_path'),
                'image_type': img_type
            }
        else:
            print(f"Warning: Could not parse: {filename}")
    
    return dict(groups)

def run_triposr_single(input_image, output_dir, 
                       no_remove_bg=True,
                       bake_texture=True, 
                       texture_resolution=512,
                       mc_resolution=0,
                       model_save_format='obj',
                       render=True,
                       verbose = False):
    """
    Run TripoSR on a single image.
    """
    subfolder = os.path.join(output_dir, '0')
    os.makedirs(subfolder, exist_ok=True)
    command = [
        'python', 'run.py', 
        input_image,
        '--output-dir', output_dir,
        '--texture-resolution', str(texture_resolution),
        '--model-save-format', model_save_format
    ]
    
    if no_remove_bg:
        command.append('--no-remove-bg')
    
    if bake_texture:
        command.append('--bake-texture')
    
    if render:
        command.append('--render')
    
    if mc_resolution !=0:
        command.append(f'--mc-resolution, {mc_resolution}')
    print(f"  Command: {' '.join(command)}")
    
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    if verbose:
        for line in process.stdout:
            print(f"    {line}", end='')
    
    process.wait()
    return process.returncode

In [7]:
def run_triposr(results, base_output_dir='./New_Result/TripoSR/',
                              no_remove_bg=True,
                              bake_texture=True,
                              texture_resolution=512,
                              mc_resolution=0,
                              model_save_format='obj',
                              render=True,
                              verbose = False):
    """
    Run TripoSR on grouped results from process_all_images().
    """
    
    os.makedirs(base_output_dir, exist_ok=True)
    
    # Group by prefix
    groups = group_results_by_prefix(results)
    
    if verbose:
        print("="*60)
        print("GROUPED RESULTS")
        print("="*60)
    for prefix, angles in groups.items():
        
        if verbose:
            print(f"\n{prefix}:")
        for angle in sorted(angles.keys()):
            # FIX: angles[angle] is a dict, get 'masked_path' from it
            masked_path = angles[angle]['masked_path']
            
            if verbose:
                print(f"  {angle:+4d} deg: {os.path.basename(masked_path)}")
    
    triposr_results = {}
    total_groups = len(groups)
    
    if verbose:
        print("\n" + "="*60)
        print(f"RUNNING TRIPOSR ON {total_groups} GROUP(S)")
        print("="*60)
    
    for group_idx, (prefix, angle_data) in enumerate(groups.items()):
        
        if verbose:
            print(f"\n{'='*60}")
            print(f"[{group_idx+1}/{total_groups}] {prefix}")
            print(f"{'='*60}")
        
        # Create group folder
        group_folder = os.path.join(base_output_dir, prefix)
        os.makedirs(group_folder, exist_ok=True)
        
        group_results = {}
        
        for angle in sorted(angle_data.keys()):
            data = angle_data[angle]
            masked_path = data['masked_path']  # Get the path string
            
            if verbose:
                print(f"\n  -- View {angle:+d} deg --")
                print(f"    Input: {os.path.basename(masked_path)}")
            
            # Create view subfolder
            view_folder = os.path.join(group_folder, f"view_{angle:+d}")
            os.makedirs(view_folder, exist_ok=True)
            
            # Copy input for reference
            shutil.copy(masked_path, os.path.join(view_folder, "input_masked.png"))
            
            # Run TripoSR
            return_code = run_triposr_single(
                input_image=masked_path,
                output_dir=view_folder,
                no_remove_bg=no_remove_bg,
                bake_texture=bake_texture,
                texture_resolution=texture_resolution,
                mc_resolution=mc_resolution,
                model_save_format=model_save_format,
                render=render,
                verbose = False
            )
            
            # Check output
            mesh_path = os.path.join(view_folder, '0', f'mesh.{model_save_format}')
            success = return_code == 0 and os.path.exists(mesh_path)
            
            group_results[angle] = {
                'input_masked': masked_path,
                'view_folder': view_folder,
                'mesh_path': mesh_path if success else None,
                'success': success
            }
            
            status = "[OK]" if success else "[FAIL]"
            print(f"    {status} {'Success' if success else 'Failed'}")
        
        triposr_results[prefix] = {
            'group_folder': group_folder,
            'views': group_results
        }
    
    # Summary
    
    if verbose:
        print("\n" + "="*60)
        print("SUMMARY")
        print("="*60)
    for prefix, data in triposr_results.items():
        
        if verbose:
            print(f"\n{prefix}/")
        for angle, view in sorted(data['views'].items()):
            status = "[OK]" if view['success'] else "[FAIL]"
            mesh = os.path.basename(view['mesh_path']) if view['mesh_path'] else "FAILED"
            
            if verbose:
                print(f"  {status} view_{angle:+d}/ -> {mesh}")
    
    return triposr_results

In [8]:
def u_gvxr_setup (bone, flesh, source_position = -50, 
                  detector_position = 50, 
                  ray_energy = 0.08, 
                  rays = 500, 
                  pixels = 1440,
                  pixel_size = 0.2,
                  verbose = False):
    # Create an OpenGL context
    
    if verbose:
        print("Create an OpenGL context")
    gvxr.createOpenGLContext()

    # Create a source
    
    if verbose:
        print("Set up the beam")
    gvxr.setSourcePosition(source_position,  0, 0, "cm");
    gvxr.useParallelBeam();
    #  For a parallel source, use gvxr.useParallelBeam();

    # Set its spectrum, here a monochromatic beam
    # 1000 photons of 80 keV (i.e. 0.08 MeV) per ray
    gvxr.setMonoChromatic(ray_energy, "MeV", rays);
    # The following is equivalent: gvxr.setMonoChromatic(80, "keV", 1000);

    # Set up the detector
    
    if verbose:
        print("Set up the detector");
    gvxr.setDetectorPosition(detector_position, 0, 0, "cm");
    gvxr.setDetectorUpVector(0, 0, -1);
    gvxr.setDetectorNumberOfPixels(pixels, pixels);
    gvxr.setDetectorPixelSize(pixel_size, pixel_size, "mm");

    # Set up the detector

    if not os.path.exists(bone):
        raise IOError(bone)

    if verbose:
        print("Load the mesh data from", bone)
    gvxr.loadMeshFile("Hand_Bone", bone, "cm")

    #print("Move ", "Hand_Bone", " to the centre")
    #gvxr.moveToCentre("Hand_Bone")
    Z  = [1, 6, 7, 8, 12, 15, 16, 20]
    w  = [0.03, 0.15, 0.04, 0.45, 0.00, 0.10, 0.00, 0.23]
    gvxr.setMixture("Hand_Bone", Z, w)
    gvxr.setDensity("Hand_Bone", 1.85, "g/cm3")   # cortical bone density

    # 3. Load and assign material to FLESH component
    
    if verbose:
        print("Load the mesh data from", flesh)
    gvxr.loadMeshFile("Flesh", flesh, "cm")
    gvxr.setCompound("Flesh", "H2O")  # Approximate soft tissue
    gvxr.setDensity("Flesh", 1.0, "g/cm3")  # Approximate soft tissue
    gvxr.resetSceneTransformation()
    gvxr.moveToCentre()
    #gvxr.rotateScene(0, 360,0)

    # Compute an X-ray image
    # We convert the array in a Numpy structure and store the data using single-precision floating-point numbers.
    print("Scene Setting done.")
    #x_ray_image = np.array(gvxr.computeXRayImage()).astype(np.single)

    # Update the visualisation window
    #gvxr.displayScene()

In [ ]:
# Old version for adding noise. Refer to the next section for noise embedding

def add_poisson_read_noise(img, gain =1, dose_scale=1e5, read_sigma=5e-4, clip=True): #https://doi.org/10.3390/s18041019
    """
    img: normalized to [0,1], higher = brighter detector signal.
    dose_scale: effective photon count scale; smaller -> noisier (low dose).
    read_sigma: Gaussian read noise std in [0,1] units.
    """
    dose_scale = dose_scale*gain
    x = np.clip(img, 0, 1)
    lam = dose_scale * x
    noisy = np.random.poisson(lam) / (dose_scale + 1e-6)
    noisy += np.random.normal(0, read_sigma, size=img.shape)
    if clip:
        noisy = np.clip(noisy, 0, 1)
    return noisy

from scipy.ndimage import gaussian_filter

def add_scatter_haze(img, gain=1, scatter_frac=0.1, sigma=15, clip=True): # DOI: 10.1016/0720-048x(91)90099-h
    """
    scatter_frac: fraction of low-frequency scatter added (0.05-0.3 typical).
    sigma: blur kernel size (pixels) controlling low-frequency nature.
    """
    scatter_frac = gain*scatter_frac
    x = np.clip(img, 0, 1)
    haze = gaussian_filter(x, sigma=sigma)
    out = (1 - scatter_frac) * x + scatter_frac * haze
    if clip:
        out = np.clip(out, 0, 1)
    return out

def add_detector_nonuniformity(img, gain = 1, gain_std=0.0005, offset_std=0.0005,
                               banding_std=0.0001, bad_pixel_frac=0.0001, clip=True):
    x = np.clip(img, 0, 1)
    H, W = x.shape

    # Gain/offset fields (low-frequency)
    gain = 1.0 + np.random.normal(0, gain_std*gain, size=(H, W))
    offset = np.random.normal(0, offset_std*gain, size=(H, W))

    # Row/column banding
    row = np.random.normal(0, banding_std*gain, size=(H, 1))
    col = np.random.normal(0, banding_std*gain, size=(1, W))

    out = gain * x + offset + row + col

    # Bad pixels (hot/dead)
    n_bad = int(bad_pixel_frac * H * W)
    if n_bad > 0:
        ys = np.random.randint(0, H, size=n_bad)
        xs = np.random.randint(0, W, size=n_bad)
        out[ys, xs] = np.random.choice([0.0, 1.0], size=n_bad)

    if clip:
        out = np.clip(out, 0, 1)
    return out

def add_beam_hardening_proxy(img, gain = 1, beta=0.05, eps=1e-6, clip=True):
    """
    beta controls nonlinearity; 0.1-0.6 typical for stress tests.
    Assumes img in [0,1], where lower intensity could mean higher attenuation.
    """
    x = np.clip(img, 0, 1)

    # Convert to a pseudo line-integral domain p ~ -log(I)
    p = -np.log(np.clip(x, eps, 1.0))
    p2 = p + beta * (p ** 2) * gain  # nonlinear thickening

    out = np.exp(-p2)
    if clip:
        out = np.clip(out, 0, 1)
    return out

def add_gaussian_psf_blur(img, gain = 1, sigma=0.4, clip=True): #https://www.aapm.org/meetings/amos2/pdf/29-7979-41789-512.pdf
    x = np.clip(img, 0, 1)
    out = gaussian_filter(x, sigma=sigma*gain)
    return np.clip(out, 0, 1) if clip else out

In [10]:
def add_poisson_read_noise_physical(x, exposure=1.0, I0=2e4, read_sigma=2e-4):
    """
    x: transmission-like image in [0,1] (higher = more photons)
    exposure: 1.0 nominal, <1.0 lower dose (more noise)
    I0: expected photons at x=1 for exposure=1
    """
    x = np.clip(x, 0, 1)

    # Photon counts (Poisson)
    lam = I0 * exposure * x
    y = np.random.poisson(lam) / (I0 * exposure + 1e-12)

    # Add read noise (Gaussian, after normalization)
    y += np.random.normal(0, read_sigma, size=x.shape)

    return y  # do NOT clip here

def clamp_percentile(x, lo=0.1, hi=99.9):
    a, b = np.percentile(x, [lo, hi])
    y = (x - a) / (b - a + 1e-12)
    return np.clip(y, 0, 1)

def degrade_acquisition(img_norm, severity=0.0, case = 0):
    x = img_norm.copy()
    if case == 1:
        # Scatter (mild)
        a = 0.15 * severity
        haze = gaussian_filter(x, sigma=30)
        x = (1-a)*x + a*haze
    elif case == 2:
        # PSF blur (system blur, mild)
        x = gaussian_filter(x, sigma=0.4*severity)

    elif case == 3:
        # Beam hardening (mild for hand)
        p = -np.log(np.clip(x, 1e-6, 1.0))
        beta = 0.05 * severity
        x = np.exp(-(p + beta*p*p))
    
    elif case == 4:
        # Poisson + read (dose)
        exposure = 1.0 - 0.7*severity   # 1.0 -> 0.3
        x = add_poisson_read_noise_physical(x, exposure=exposure, I0=2e5, read_sigma=1e-5)

    elif case == 0:
        # Scatter (mild)
        a = 0.08 * severity
        haze = gaussian_filter(x, sigma=20)
        x = (1-a)*x + a*haze

        # PSF blur (system blur, mild)
        x = gaussian_filter(x, sigma=0.4*severity)

        # Beam hardening (mild for hand)
        p = -np.log(np.clip(x, 1e-6, 1.0))
        beta = 0.03 * severity
        x = np.exp(-(p + beta*p*p))

        # Poisson + read (dose)
        exposure = 1.0 - 0.6*severity   # 1.0 -> 0.3
        x = add_poisson_read_noise_physical(x, exposure=exposure, I0=2e5, read_sigma=1e-5)

    else:
        return x

    # Single clamp (avoid saturation)
    x = clamp_percentile(x, lo=0.1, hi=99.9)
    return x


In [11]:
# Create the output directory if needed
def add_scintillator_noise(x_ray_image, noise_intensity=0.05, grain_size=1):
    """
    Add paper-like scintillator texture to synthetic X-ray images.
    
    Parameters:
    - x_ray_image: Input image array
    - noise_intensity: Strength of the noise (0.02-0.1 typical)
    - grain_size: Size of noise grains (1 = pixel-level, higher = coarser)
    """
    from scipy.ndimage import gaussian_filter
    
    # Create base noise pattern
    noise = np.random.normal(0, 1, x_ray_image.shape)
    
    # Apply slight blur to simulate grain structure
    if grain_size > 1:
        noise = gaussian_filter(noise, sigma=grain_size)
    
    # Scale noise relative to local image intensity
    # (scintillator noise is often signal-dependent)
    img_normalized = (x_ray_image - x_ray_image.min()) / (x_ray_image.max() - x_ray_image.min() + 1e-6)
    
    # Add multiplicative + additive noise components
    noisy_image = img_normalized * (1 + noise_intensity * noise) + noise_intensity * noise * x_ray_image.mean()
    
    return noisy_image


def gamma_imaging (x_ray_image, output_path, gamma=0.5, figsize=(8, 8), plotting = False, noise_input = (0, 1), case = 0):
    """
    Enhanced X-ray with gamma correction.
    """ 
    epsilon = 1e-5
    noise, grain_size = noise_input
    gain = noise_input[0]
    # Normalize
    #gain = 0.8
    img_norm = (x_ray_image - x_ray_image.min()) / (x_ray_image.max() - x_ray_image.min() + epsilon)
    img_norm = np.clip(img_norm, 0, 1)
    #img_norm = add_detector_nonuniformity(img_norm, gain=gain)
    if gain != 0:
        img_norm= degrade_acquisition(img_norm, gain, case)
        '''img_norm = add_scatter_haze(img_norm, gain=gain)
        img_norm = add_gaussian_psf_blur(img_norm, gain=gain)
        img_norm = add_poisson_read_noise(img_norm, gain=gain)
        img_norm = add_beam_hardening_proxy(img_norm, gain=gain)'''
    # Apply log for better dynamic range
    #epsilon = 1e-8
    img_log = np.log(img_norm + epsilon)
    #img_log_norm = (img_log - img_log.min()) / (img_log.max() - img_log.min() + epsilon)
    img_log_norm = (img_log - np.log(epsilon)) / (0 - np.log(epsilon))
    img_log_norm = np.clip(img_log_norm, 0, 1)
    
    # Apply gamma correction (gamma < 1 brightens mid-tones)
    img_gamma = np.power(img_log_norm, gamma)
    
    # Invert so bone (thick) = dark, air/thin = bright
    img_final = 1.0 - img_gamma
    if noise != 0:
        noise = gain*0.05
        fig, ax = plt.subplots(figsize=figsize)
        ax.imshow(1-img_final, cmap='gray_r')
        ax.axis('off')
        plt.savefig(output_path[:-4]+'_ori.png', bbox_inches='tight', pad_inches=0, dpi=150)
        if ~plotting:
            plt.close(fig)
    
    
    
    # Save
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(img_final, cmap='gray_r')
    ax.axis('off')
    plt.savefig(output_path, bbox_inches='tight', pad_inches=0, dpi=150)
    if ~plotting:
        plt.close(fig)
    

    return img_final

def u_gvxr_start (output_folder, 
                  source_position = -50, 
                  detector_position = 50, 
                  viewing_angles = [0, 30, -30], 
                  output_num = 0, show_plot = False, verbose = False, noise = (0, 1), Rotation = [0,0,0], case = 0):

    image_paths=[]
    image_names=[]
    original_image_paths=[]
    normalized_image_paths=[]
    logNorm_image_paths=[]
    powerNorm_image_paths=[]
    gamma_image_paths = []
    
    for angle in viewing_angles:
        if not os.path.exists("output_data"):
            os.mkdir("output_data")
        

        # Capture projections
        setup_gantry_at_angle(angle, source_position, detector_position, verbose= verbose, scene_rotate=Rotation)
        x_ray_image = np.array(gvxr.computeXRayImage()).astype(np.float32)
        # Save the X-ray image in a TIFF file and store the data using single-precision floating-point numbers.
        image_path, image_name = file_name(f"gvxr_processed_{angle}", output_folder, '.tif')
        image_paths.append(image_path)
        image_names.append(image_name)

        imwrite(image_path, x_ray_image)
        if verbose:
            print(f"Saved TIFF: {image_path}")

        # The line below will also works
        # imwrite('output_data/raw_x-ray_image-02.tif', x_ray_image)

        # Save the L-buffer
        #gvxr.saveLastLBuffer('output_data/lbuffer-02.tif');

        # Display the X-ray image
        # using a linear colour scale

        # Open the .tif file
        tif_image = Image.open(image_path)
        converted_image = tif_image.convert('I;16')
        converted_image.save(image_path[:-4]+'.png')

        # Convert 'F' mode to numpy array, normalize, then back to 8-bit image
        image_array = np.array(tif_image)

        # Normalize to 0-255
        normalized = (255 * (image_array - image_array.min()) / (image_array.ptp())).astype(np.uint8)

        # Convert back to image in 'L' mode
        normalized_image = Image.fromarray(normalized, mode='L')
        normalized = (image_array - image_array.min()) / (image_array.ptp())

        # Apply power-law transformation (gamma)
        gamma = 0.5  # Choose your gamma, e.g., 0.5 for enhancement
        gamma_corrected = np.power(normalized, gamma)

        # Scale to 0-255 and convert to uint8
        scaled = (255 * gamma_corrected).astype(np.uint8)

        image_lists = [image_path[:-4]+'ori.png', image_path[:-4]+'norm.png', image_path[:-4]+'Log.png', image_path[:-4]+'Power.png', image_path[:-4]+'gamma.png']
        # Convert to PIL image and save
        output_image = Image.fromarray(scaled, mode='L')

        output_image.save(image_lists[0])
        # Save as PNG
        normalized_image.save(image_lists[1])
        # Save as .png
        #tif_image.save("output_data/raw_x-ray_image-03.png")
        
        fig, ax = plt.subplots(figsize=(8, 8))
        
        ax.imshow(x_ray_image if noise[0] ==0 else add_scintillator_noise(x_ray_image, noise_intensity=noise[0], grain_size=noise[1]), norm=LogNorm(), cmap="gray_r") 
        ax.axis('off')
        plt.savefig(image_lists[2], bbox_inches='tight', pad_inches=0)
        if ~show_plot:
            plt.close(fig)

        fig, ax = plt.subplots(figsize=(8, 8))
        ax.imshow(x_ray_image if noise[0] ==0 else add_scintillator_noise(x_ray_image, noise_intensity=noise[0], grain_size=noise[1]), norm=PowerNorm(gamma=0.8/2.), cmap="gray_r") 
        ax.axis('off')
        plt.savefig(image_lists[3], bbox_inches='tight', pad_inches=0)
        if ~show_plot:
            plt.close(fig)
        
        gamma_imaging(x_ray_image,image_lists[4], gamma = gamma, plotting = show_plot, noise_input = noise, case = case)

        original_image_paths.append(image_lists[0])
        normalized_image_paths.append(image_lists[1])
        logNorm_image_paths.append(image_lists[2])
        powerNorm_image_paths.append(image_lists[3])
        gamma_image_paths.append(image_lists[4])

    
    if output_num ==0:
        return original_image_paths
    elif output_num ==1:
        return normalized_image_paths
    elif output_num ==2:
        return logNorm_image_paths
    elif output_num ==3:
        return powerNorm_image_paths
    elif output_num ==4:
        return gamma_image_paths
    else:
        return

In [12]:
def setup_gvxr_image_save_triposr (bone, flesh, output_folder, viewing_angles = [0, 30, -30],
        MASK_DIR = "./New_Result/Mask/",
        ORIGINAL_DIR = "./New_Result/Original/",
        PREPROCESS_DIR = "./New_Result/Preprocessed/",
        PROMPT_DIR = "./New_Result/Prompts/",
        MASKED_DIR = "./New_Result/Masked/",
        BASE_OUTPUT_DIR= './New_Result/TripoSR/',
        source_position = -50, 
        detector_position = 50, 
        ray_energy = 0.08, 
        rays = 500, 
        pixels = 1440,
        output_num = 0,
        mask_num = 1,
        pixel_size = 0.2,
        show_plots = False,
        verbose = False,
        noise = (0,1),
        Rotation = [0,0,0],
        case = 0):

    u_gvxr_setup (bone, flesh, source_position = source_position, 
                    detector_position = detector_position, 
                    ray_energy = ray_energy, 
                    rays = rays, 
                    pixels = pixels,
                    pixel_size = pixel_size,
                    verbose = verbose)
    if verbose:
        print(source_position)
    original_image_paths= u_gvxr_start (output_folder, 
                                        source_position = source_position, 
                                        detector_position = detector_position, 
                                        viewing_angles = viewing_angles,
                                        output_num = output_num,
                                        noise = noise,
                                        verbose = verbose,
                                        Rotation=Rotation,
                                        case = case)
    # Process all images

    results = process_all_images(
        original_image_paths=original_image_paths,
        pose_model=pose_model,
        predictor=predictor,
        MASK_DIR=MASK_DIR,
        ORIGINAL_DIR=ORIGINAL_DIR,
        PREPROCESS_DIR=PREPROCESS_DIR,
        PROMPT_DIR=PROMPT_DIR,
        MASKED_DIR=MASKED_DIR,
        mask_num=mask_num,
        show_plots=show_plots,  # Set True to see plots for each image
        verbose = verbose
    )

    # Access results for specific images
    if verbose:
        for path, result in results.items():
            print(f"\n{os.path.basename(path)}:")
            print(f"  Masked image: {result['masked_path']}")
    
    triposr_results = run_triposr(
        results,  # <-- Your results dict from process_all_images()
        base_output_dir= BASE_OUTPUT_DIR,
        no_remove_bg=False,
        bake_texture=False,
        texture_resolution=512,
        model_save_format='obj',
        render=False,
        verbose = verbose
    )

In [13]:
import random
def generate_random_angles(n_angles=3, min_angle=-180, max_angle=180, min_separation=5):

    max_attempts = 1000
    
    for _ in range(max_attempts):
        angles = []
        for _ in range(n_angles):
            angle = random.randint(min_angle, max_angle)
            angles.append(angle)
        
        # Check minimum separation
        angles_sorted = sorted(angles)
        valid = True
        for i in range(len(angles_sorted) - 1):
            if abs(angles_sorted[i+1] - angles_sorted[i]) < min_separation:
                valid = False
                break
        
        if valid:
            return angles_sorted
    
    # Fallback: evenly spaced angles
    return list(np.linspace(min_angle, max_angle, n_angles, dtype=int))

def cleanup_gvxr(verbose = False):
    """Complete cleanup of gVXR resources"""
    try:
        # Remove all meshes
        gvxr.removePolygonMeshesFromSceneGraph()
        if verbose:
            print("  [OK] Removed all meshes")
    except Exception as e:
        print(f"  Warning: Could not remove meshes: {e}")
    
    try:
        # Reset beam
        gvxr.resetBeamSpectrum()
        if verbose:
            print("  [OK] Reset beam spectrum")
    except Exception as e:
        print(f"  Warning: Could not reset beam: {e}")
    
    try:
        # Reset scene
        gvxr.resetSceneTransformation()
        if verbose:
            print("  [OK] Reset scene transformation")
    except Exception as e:
        print(f"  Warning: Could not reset scene: {e}")
    
    try:
        # Destroy OpenGL context
        gvxr.destroyOpenGLContext()
        if verbose:
            print("  [OK] Destroyed OpenGL context")
    except Exception as e:
        print(f"  Warning: Could not destroy context: {e}")
    if verbose:
        print("gVXR cleanup complete")

In [14]:
import psutil
import gc
bone = ["./Bone_V4.stl", "./Bone_V4.stl", "./Bone_V5.stl"]
flesh = ["./Flesh_V4_bool.stl", "./Flesh_V4_bool.stl", "./Flesh_V5_bool.stl"]
n_iterations = 3
memory_threshold_gb = 32
for j in range(3):#,0.8,0.9, 1]:
        output_folder =f"./output_gvxr_noise_output_model_PSF_{j}/"
        for i in list(range(-180, -140)) + list(range(-20, -10)) + list(range(10, 20))+list(range(140, 150)):#[180]:#list(range(-45, 45)):

                if i % 5 == 0:
                        process = psutil.Process(os.getpid())
                        memory_gb = process.memory_info().rss / 1e9
                        print(f"Memory: {memory_gb:.1f} GB")

                        if memory_gb > memory_threshold_gb:
                                print("Memory too high! Performing cleanup...")
                                gc.collect()
                                if torch.cuda.is_available():
                                        torch.cuda.empty_cache()
                # Generate random angles
                #angles = generate_random_angles()
                try:
                        setup_gvxr_image_save_triposr (bone[j], flesh[j], output_folder, viewing_angles = np.array([i]),
                                MASK_DIR = "./New_Result/Mask5/",
                                ORIGINAL_DIR = "./New_Result/Original5/",
                                PREPROCESS_DIR = "./New_Result/Preprocessed5/",
                                PROMPT_DIR = "./New_Result/Prompts5/",
                                MASKED_DIR = "./New_Result/Masked5/",
                                BASE_OUTPUT_DIR= f'./New_Result/TripoSR_figure_0.06_model_PSF_{j}/',
                                source_position = -80, 
                                detector_position = 80, 
                                ray_energy = 0.06, 
                                rays = 2000, 
                                pixels = 1440,
                                output_num = 4,
                                mask_num =1,
                                pixel_size = 1.5,
                                show_plots = False,
                                verbose = False,
                                noise = (1, 1),
                                Rotation = [0,0,0],
                                case = 2)
                        print(f"[{i+1}/{n_iterations} SUCCESS with Angles: {i}")
                
                except Exception as e:
                        print(f"Angles: {i}  ERROR: {str(e)}")

                finally:
                        cleanup_gvxr()
                        gc.collect()
                
                        if torch.cuda.is_available():
                                torch.cuda.empty_cache()

cleanup_gvxr()
gc.collect()
if torch.cuda.is_available():
        torch.cuda.empty_cache()

Memory: 2.8 GB
Scene Setting done.


C:\Users\Byungjoon_Bae\AppData\Local\Temp\ipykernel_52640\4253817714.py:134: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  normalized_image = Image.fromarray(normalized, mode='L')
C:\Users\Byungjoon_Bae\AppData\Local\Temp\ipykernel_52640\4253817714.py:146: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  output_image = Image.fromarray(scaled, mode='L')
C:\Users\Byungjoon_Bae\AppData\Local\Temp\ipykernel_52640\2485339320.py:147: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  masked_image_pil = Image.fromarray(rgba_image, mode='RGBA')


  Command: python run.py ./New_Result/Masked5/000002_gvxr_processed_-180_gamma_masked.png --output-dir ./New_Result/TripoSR_figure_0.06_model_PSF_0/000002_gvxr_processed\view_-180 --texture-resolution 512 --model-save-format obj
    [OK] Success
[-179/3 SUCCESS with Angles: -180
Scene Setting done.
  Command: python run.py ./New_Result/Masked5/000002_gvxr_processed_-179_gamma_masked.png --output-dir ./New_Result/TripoSR_figure_0.06_model_PSF_0/000002_gvxr_processed\view_-179 --texture-resolution 512 --model-save-format obj
    [OK] Success
[-178/3 SUCCESS with Angles: -179
Scene Setting done.
  Command: python run.py ./New_Result/Masked5/000002_gvxr_processed_-178_gamma_masked.png --output-dir ./New_Result/TripoSR_figure_0.06_model_PSF_0/000002_gvxr_processed\view_-178 --texture-resolution 512 --model-save-format obj
    [OK] Success
[-177/3 SUCCESS with Angles: -178
Scene Setting done.
  Command: python run.py ./New_Result/Masked5/000002_gvxr_processed_-177_gamma_masked.png --output-d